In [25]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
import numpy as np

# фиксируем сиды
torch.manual_seed(42)
np.random.seed(42)

# Генерация данных

In [26]:
n_samples = 10_000  # общее кол-во экземпляров
embedding_dim = 64  # размер эмбеддинга
noise_std = 0.05  # уровень шума

X = torch.randn(n_samples, embedding_dim) * 2
w_true = torch.randn(embedding_dim, 1) * 0.7
b_true = torch.tensor([0.7])

# таргет = sigmoid(linear + noise) -> значения в (0,1)
linear = X @ w_true + b_true
y = torch.sigmoid(linear + noise_std * torch.randn_like(linear)).squeeze(1)

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X.numpy(), y.numpy(), test_size=0.2, random_state=42
)
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

# нормализация признаков
mean = X_train.mean(0, keepdim=True)
std  = X_train.std(0, keepdim=True) + 1e-6
X_train_norm = (X_train - mean) / std
X_test_norm  = (X_test  - mean) / std

In [27]:
print(f'X_train_norm size:\t{X_train_norm.size()}')
print(f'y_train size:\t\t{y_train.size()}\n')

print(f'X_test_norm size:\t{X_test_norm.size()}')
print(f'y_test size:\t\t{y_test.size()}')

X_train_norm size:	torch.Size([8000, 64])
y_train size:		torch.Size([8000, 1])

X_test_norm size:	torch.Size([2000, 64])
y_test size:		torch.Size([2000, 1])


In [28]:
torch.save(X_train, '../data/split/worker1/x_train.pth')
torch.save(X_train_norm, '../data/split/worker1/x_train_norm.pth')
torch.save(X_test, '../data/split/worker1/x_test.pth')
torch.save(X_test_norm, '../data/split/worker1/x_test_norm.pth')

torch.save(y_train, '../data/split/worker2/y_train.pth')
torch.save(y_test, '../data/split/worker2/y_test.pth')

In [29]:
torch.save(X_train_norm[:2000], '../data/aug/x_train_norm_worker1.pth')
torch.save(y_train[:2000], '../data/aug/y_train_worker1.pth')

torch.save(X_train_norm[2000:3200], '../data/aug/x_train_norm_worker2.pth')
torch.save(y_train[2000:3200], '../data/aug/y_train_worker2.pth')

torch.save(X_test_norm[:800], '../data/aug/x_test_norm_worker1.pth')
torch.save(y_test[:800], '../data/aug/y_test_worker1.pth')

# SGD

## Model

In [30]:
class LinearRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.linear = nn.Linear(input_dim, 1)
    def forward(self, x):
        return self.linear(x)

model = LinearRegressor(embedding_dim)
nn.init.normal_(model.linear.weight, mean=0.0, std=0.01)
nn.init.constant_(model.linear.bias, 0.0)

criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.9)

## Обучение

In [36]:
t0 = time.time()

batch_size = 512
n_epochs = 30
dataset = torch.utils.data.TensorDataset(X_train_norm, y_train)
loader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=True)

for epoch in range(n_epochs):
    model.train()
    for xb, yb in loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
    if (epoch+1) % 5 == 0 or epoch == 0:
        print(f"[Epoch {epoch+1:02d}] loss={loss:.6f}")
        
print(f"Elapsed time: {time.time() - t0:.4f} seconds")

[Epoch 01] loss=0.061602
[Epoch 05] loss=0.064139
[Epoch 10] loss=0.061429
[Epoch 15] loss=0.063065
[Epoch 20] loss=0.058993
[Epoch 25] loss=0.058623
[Epoch 30] loss=0.061214
Elapsed time: 1.8394 seconds


# Валидация

In [37]:
def metrics(y_true, y_pred):
    """Расчет метрик"""
    mse = torch.mean((y_true - y_pred) ** 2).item()
    return mse

In [38]:
model.eval()
with torch.no_grad():
    y_pred_sgd = model(X_test_norm)

mse_s = metrics(y_test, y_pred_sgd)
print(f"SGD (MSELoss):   MSE={mse_s:.6f}")

SGD (MSELoss):   MSE=0.066915


In [34]:
model.linear.weight

Parameter containing:
tensor([[ 0.0223,  0.0192,  0.0514, -0.0907,  0.0087, -0.0815, -0.1133, -0.0118,
          0.0281,  0.0614,  0.0741,  0.0483, -0.0120,  0.0600, -0.1338, -0.0106,
          0.0751, -0.0540,  0.0435,  0.0194,  0.0507, -0.0221,  0.0268,  0.0125,
         -0.0651,  0.0648,  0.0027, -0.0125, -0.0267, -0.0436, -0.0695,  0.0391,
          0.0399,  0.0772, -0.0400,  0.0145,  0.0477,  0.0751, -0.0291, -0.0102,
          0.0431, -0.0525, -0.0782,  0.0404, -0.0214,  0.0068, -0.0600, -0.0436,
          0.0284, -0.0427,  0.0156, -0.0098,  0.0102, -0.0016,  0.0202, -0.0219,
         -0.0680, -0.0640,  0.0368, -0.0315, -0.0159,  0.0523, -0.0905,  0.0327]],
       requires_grad=True)

In [35]:
model.linear.bias

Parameter containing:
tensor([0.5071], requires_grad=True)